# 01. Data Overview

## XAI 기반 OTT 신규 고객 이탈 요인 분석 및 리텐션 전략

이 노트북은 `park.ingyeom` 작업 공간에서 실행할 첫 번째 데이터 점검 노트북입니다.

`00_project_plan.ipynb`가 프로젝트의 문제 정의와 분석 원칙을 고정하는 문서였다면, 이 노트북은 실제 원본 CSV들이 그 문제 정의에 맞게 존재하는지, 키 구조가 어떻게 연결되는지, 브리핑에서 확인한 주요 수치가 재현되는지를 확인합니다.

이 단계에서는 아직 본격적인 전처리나 파생변수 생성을 수행하지 않습니다. 여기서 하는 일은 다음 네 가지입니다.

1. 원본 파일이 정상적으로 로딩되는지 확인한다.
2. 각 파일의 행 수, 컬럼, 주요 key를 확인한다.
3. `100원딜 = is_promotion = 1 = price 100` 관계가 성립하는지 확인한다.
4. 이후 노트북에서 사용할 전처리 기준의 출발점이 되는 수치를 검산한다.

## 1-1. 이 노트북의 역할

이 노트북은 분석 전체 파이프라인에서 다음 위치에 있습니다.

```text
00_project_plan.ipynb
  프로젝트 목적, 문제 정의, 가설, 분석 원칙 고정

01_data_overview.ipynb
  원본 데이터 구조 확인, key 관계 확인, 주요 브리핑 수치 재현

02_preprocessing_policy.ipynb
  더미 이상치 제거, 3주 관측창 정의, 분석 단위 확정

03_movie_metadata_unification.ipynb
  Movie_Master 기준 Wavve/KOBIS 메타데이터 통합

04_usage_feature_engineering.ipynb
  View_History 기반 시청 행동 파생변수 생성

05_content_feature_engineering.ipynb
  영화 메타데이터 기반 유저 콘텐츠 성향 변수 생성
```

따라서 01번의 최종 산출물은 모델링용 데이터가 아닙니다. 이 노트북의 산출물은 데이터 구조 점검표와 이후 단계에서 참고할 검산 요약입니다.

## 1-2. 입력 파일과 출력 파일

### 입력 파일

원본 파일은 `park.ingyeom/_data/01_raw/`에 있다고 가정합니다.

```text
park.ingyeom/_data/01_raw/
├─ Membership_v1.csv
├─ User_Mapping_v1.csv
├─ View_History_v1.csv
├─ Movie_Master_v1.csv
├─ wavve_movies_filtered_by  정규식 (1)(1).csv
└─ wavve_notfound_kobis_filtered_by_char_match(1).csv
```

### 출력 파일

이 노트북은 다음 파일을 생성합니다.

```text
park.ingyeom/reports/tables/01_data_overview_file_summary.csv
park.ingyeom/reports/tables/01_data_overview_column_overview.csv
park.ingyeom/reports/tables/01_data_overview_key_checks.csv
park.ingyeom/reports/tables/01_data_overview_metadata_coverage.csv
```

이 출력 파일들은 이후 단계의 입력 데이터가 아니라, 원본 데이터 상태를 기록하는 감사 로그에 가깝습니다.

In [ ]:
from pathlib import Path
import re
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# 이 노트북은 park.ingyeom 폴더에서 실행하는 것을 권장합니다.
# 예: .../ott-churn-prediction/park.ingyeom

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]

    for path in candidates:
        if path.name == "park.ingyeom":
            return path
        if (path / "park.ingyeom").exists():
            return path / "park.ingyeom"

    # 로컬 테스트나 임시 실행 환경에서는 현재 위치를 루트로 사용합니다.
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "_data"
RAW_DATA_DIR = DATA_DIR / "01_raw"
INTERIM_DATA_DIR = DATA_DIR / "02_interim"
PROCESSED_DATA_DIR = DATA_DIR / "03_processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"

for path in [RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("TABLES_DIR:", TABLES_DIR)

## 1-3. 원본 파일 경로 설정

파일명은 현재 프로젝트에서 사용한 실제 파일명을 기준으로 둡니다.

주의할 점은 Wavve 파일명에 공백과 한글이 들어 있다는 것입니다. Git이나 운영체제 환경에 따라 파일명이 바뀌면 아래 `FILE_NAMES`만 수정하면 됩니다.

In [ ]:
FILE_NAMES = {
    "membership": "Membership_v1.csv",
    "mapping": "User_Mapping_v1.csv",
    "view": "View_History_v1.csv",
    "movie_master": "Movie_Master_v1.csv",
    "wavve": "wavve_movies_filtered_by  정규식 (1)(1).csv",
    "kobis": "wavve_notfound_kobis_filtered_by_char_match(1).csv",
}

# Colab, 임시 폴더, 로컬 복사본 등에서 실행할 가능성을 고려해 후보 폴더를 여럿 둡니다.
SEARCH_DIRS = [
    RAW_DATA_DIR,
    PROJECT_ROOT / "_data" / "raw",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
    Path.cwd(),
    Path("/mnt/data"),
]

def resolve_file(file_name: str) -> Path:
    for folder in SEARCH_DIRS:
        candidate = folder / file_name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"파일을 찾지 못했습니다: {file_name}")

PATHS = {name: resolve_file(file_name) for name, file_name in FILE_NAMES.items()}

for name, path in PATHS.items():
    print(f"{name:>12}: {path}")

## 1-4. 데이터 로딩

이 셀에서는 원본 CSV를 그대로 읽습니다.

전처리는 아직 하지 않습니다. 날짜 파싱이나 컬럼명 보정도 필요한 최소 범위에서만 수행합니다. 전처리 정책은 `02_preprocessing_policy.ipynb`에서 확정합니다.

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

membership = read_csv(PATHS["membership"])
user_mapping = read_csv(PATHS["mapping"])
view_history = read_csv(PATHS["view"])
movie_master = read_csv(PATHS["movie_master"])
wavve_meta = read_csv(PATHS["wavve"])
kobis_meta = read_csv(PATHS["kobis"])

datasets = {
    "membership": membership,
    "user_mapping": user_mapping,
    "view_history": view_history,
    "movie_master": movie_master,
    "wavve_meta": wavve_meta,
    "kobis_meta": kobis_meta,
}

path_lookup = {
    "membership": PATHS["membership"],
    "user_mapping": PATHS["mapping"],
    "view_history": PATHS["view"],
    "movie_master": PATHS["movie_master"],
    "wavve_meta": PATHS["wavve"],
    "kobis_meta": PATHS["kobis"],
}

file_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "path": str(path_lookup[name]),
    }
    for name, df in datasets.items()
])

display(file_summary)
file_summary.to_csv(TABLES_DIR / "01_data_overview_file_summary.csv", index=False, encoding="utf-8-sig")

### 현재 데이터에서 기대되는 행 수

현재 대화에서 확인한 원본 데이터 기준 기대값은 다음과 같습니다.

| 데이터셋 | 기대 행 수 | 기대 컬럼 수 |
|---|---:|---:|
| Membership_v1.csv | 17,876 | 15 |
| User_Mapping_v1.csv | 19,877 | 2 |
| View_History_v1.csv | 106,205 | 5 |
| Movie_Master_v1.csv | 14,018 | 3 |
| Wavve metadata | 4,060 | 31 |
| KOBIS metadata | 999 | 13 |

이 수치가 다르면 파일 버전이 바뀌었거나, 다른 전처리본을 읽고 있을 가능성이 있습니다.

## 1-5. 컬럼 구조 확인

각 데이터셋의 컬럼을 확인합니다. 이 단계의 목적은 모델링이 아니라, 이후 노트북에서 사용할 key와 주요 변수를 명확히 하는 것입니다.

In [ ]:
column_overview = []
for name, df in datasets.items():
    for col in df.columns:
        column_overview.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_rate": float(df[col].isna().mean()),
            "n_unique": int(df[col].nunique(dropna=True)),
        })

column_overview = pd.DataFrame(column_overview)
display(column_overview)
column_overview.to_csv(TABLES_DIR / "01_data_overview_column_overview.csv", index=False, encoding="utf-8-sig")

## 1-6. Membership 기본 점검

`Membership_v1.csv`는 이 프로젝트의 중심 테이블입니다.

| 컬럼 | 의미 |
|---|---|
| `USER_KEY` | 멤버십 기준 고객 식별자 |
| `is_repurchase` | 재구독 여부, target |
| `is_promotion` | 프로모션 여부 |
| `price` | 결제 가격 |
| `max_screen` | 동시 접속 또는 요금제 screen 수 |
| `is_churn_prevented` | 이탈 방어 조치 여부 |
| `is_user_verified` | 본인 인증 여부 |
| `gender`, `age` | 성별, 연령 |
| `reg_date`, `end_date` | 구독 시작일, 종료일 |

In [ ]:
print("Membership shape:", membership.shape)
print("USER_KEY unique:", membership["USER_KEY"].nunique())
print("is_repurchase value counts:")
display(membership["is_repurchase"].value_counts(dropna=False).to_frame("count"))

repurchase_summary = pd.DataFrame({
    "metric": ["overall_repurchase_rate", "overall_churn_rate"],
    "value": [membership["is_repurchase"].mean(), 1 - membership["is_repurchase"].mean()],
})

display(repurchase_summary)

In [ ]:
# 100원딜과 is_promotion의 관계 확인
membership_check = membership.copy()
membership_check["price_100_flag"] = (membership_check["price"] == 100).astype(int)

promo_price_crosstab = pd.crosstab(
    membership_check["is_promotion"],
    membership_check["price_100_flag"],
    rownames=["is_promotion"],
    colnames=["price_100_flag"],
)

promo_summary = membership_check.groupby("is_promotion")["is_repurchase"].agg(["count", "mean"]).reset_index()
promo_summary = promo_summary.rename(columns={"mean": "repurchase_rate"})

print("price == 100 과 is_promotion == 1의 일치율:", (membership_check["price_100_flag"] == membership_check["is_promotion"]).mean())
display(promo_price_crosstab)
display(promo_summary)

### 100원딜 정의에 대한 현재 판단

현재 데이터에서는 `price == 100`과 `is_promotion == 1`이 완전히 일치하는 것으로 확인되었습니다.

따라서 이후 분석에서는 다음 정의를 사용할 수 있습니다.

```text
100원딜 고객 = price == 100 = is_promotion == 1
비100원딜 고객 = price != 100 = is_promotion == 0
```

단, 이 정의는 현재 제공된 데이터 버전에 대한 판단입니다. 다른 버전의 데이터가 들어오면 반드시 이 교차표를 다시 확인해야 합니다.

## 1-7. 더미 이상치 후보 확인

기존 브리핑에서는 `gender = N`, `is_user_verified = 0`, `age = 40`이 동시에 나타나는 케이스를 더미 인구통계값으로 의심했습니다.

이 노트북에서는 해당 케이스의 개수와 프로모션 여부 분포만 확인합니다. 실제 제거는 `02_preprocessing_policy.ipynb`에서 수행합니다.

In [ ]:
dummy_mask = (
    (membership["gender"] == "N")
    & (membership["is_user_verified"] == 0)
    & (membership["age"] == 40)
)

dummy_summary = pd.DataFrame({
    "metric": ["dummy_anomaly_rows", "dummy_anomaly_rate"],
    "value": [int(dummy_mask.sum()), float(dummy_mask.mean())],
})

print("더미 이상치 후보 개수:", int(dummy_mask.sum()))
display(dummy_summary)
display(pd.crosstab(dummy_mask, membership["is_promotion"], rownames=["dummy_anomaly"], colnames=["is_promotion"]))
display(membership.loc[dummy_mask].groupby("is_promotion")["is_repurchase"].agg(["count", "mean"]))

### 더미 이상치 후보에 대한 현재 판단

현재 데이터에서는 `gender = N`, `is_user_verified = 0`, `age = 40` 조건에 해당하는 행이 2,639개이며, 모두 비프로모션 집단에 있습니다.

이 패턴은 실제 40대 미인증 고객이라기보다, 본인인증을 하지 않은 고객에게 기본 인구통계값이 채워진 경우일 가능성이 높습니다.

따라서 02번 전처리 노트북에서는 이 조건을 분석용 데이터에서 제거하는 정책을 적용합니다.

## 1-8. User_Mapping key 구조 확인

`User_Mapping_v1.csv`는 `Membership.USER_KEY`와 `View_History.USER_NUM`을 연결하는 브릿지 테이블입니다.

이 프로젝트에서는 분석 단위를 개인 생애 단위가 아니라 구독 이벤트 단위로 봅니다. 따라서 `USER_KEY`가 여러 `USER_NUM`과 연결될 가능성은 오류라기보다 복수 계정, 재가입, 계정 변경의 흔적으로 해석할 수 있습니다.

다만 key 중복은 join 결과의 행 수를 바꿀 수 있으므로 반드시 기록합니다.

In [ ]:
mapping_checks = {
    "mapping_rows": len(user_mapping),
    "mapping_user_key_unique": user_mapping["USER_KEY"].nunique(),
    "mapping_user_num_unique": user_mapping["USER_NUM"].nunique(),
    "duplicated_user_key_rows": int(user_mapping["USER_KEY"].duplicated(keep=False).sum()),
    "duplicated_user_key_unique": int(user_mapping.loc[user_mapping["USER_KEY"].duplicated(keep=False), "USER_KEY"].nunique()),
    "duplicated_user_num_rows": int(user_mapping["USER_NUM"].duplicated(keep=False).sum()),
    "duplicated_user_num_unique": int(user_mapping.loc[user_mapping["USER_NUM"].duplicated(keep=False), "USER_NUM"].nunique()),
}

mapping_checks_df = pd.DataFrame([mapping_checks]).T.reset_index()
mapping_checks_df.columns = ["metric", "value"]
display(mapping_checks_df)

# Membership와 mapping을 USER_KEY 기준으로 붙였을 때 행 수 변화 확인
membership_mapping_join = membership.merge(user_mapping, on="USER_KEY", how="left", indicator=True)
join_summary = pd.DataFrame({
    "metric": [
        "membership_rows_before_join",
        "rows_after_left_join_with_mapping",
        "matched_rows",
        "left_only_rows",
    ],
    "value": [
        len(membership),
        len(membership_mapping_join),
        int((membership_mapping_join["_merge"] == "both").sum()),
        int((membership_mapping_join["_merge"] == "left_only").sum()),
    ],
})

display(join_summary)

### User_Mapping에 대한 현재 판단

`USER_KEY` 중복은 모델링 전에 반드시 인지해야 하지만, 현재 프로젝트에서는 무조건 제거할 오류로 보지 않습니다.

이유는 분석 단위가 구독 이벤트이며, 같은 사람이 복수 계정을 만들거나 재가입했을 가능성이 있기 때문입니다.

다만 이후 노트북에서 join을 할 때 행 수가 증가하는지 계속 검산해야 합니다.

## 1-9. View_History 기본 구조 확인

`View_History_v1.csv`는 고객의 시청 행동을 담고 있습니다.

| 컬럼 | 의미 |
|---|---|
| `USER_NUM` | 시청이력 기준 유저 번호 |
| `MOVIE_NUM` | 영화 번호 |
| `watch_time(min)` | 시청시간, 분 단위 |
| `watch_day` | 시청일 |
| `watch_seq` | 시청 순번 |

이 파일은 이후 04번 노트북에서 1~3주차 행동 파생변수로 변환됩니다.

In [ ]:
view = view_history.copy()
view["watch_day_dt"] = pd.to_datetime(view["watch_day"].astype(str), format="%Y%m%d", errors="coerce")

view_summary = pd.DataFrame({
    "metric": [
        "view_rows",
        "unique_user_num",
        "unique_movie_num",
        "watch_day_min",
        "watch_day_max",
        "total_watch_time_min",
        "mean_watch_time_min",
        "median_watch_time_min",
    ],
    "value": [
        len(view),
        view["USER_NUM"].nunique(),
        view["MOVIE_NUM"].nunique(),
        str(view["watch_day_dt"].min().date()),
        str(view["watch_day_dt"].max().date()),
        float(view["watch_time(min)"].sum()),
        float(view["watch_time(min)"].mean()),
        float(view["watch_time(min)"].median()),
    ],
})

display(view_summary)

In [ ]:
# View_History의 USER_NUM이 mapping에 얼마나 존재하는지 확인합니다.
view_user_nums = set(view_history["USER_NUM"].dropna().unique())
mapping_user_nums = set(user_mapping["USER_NUM"].dropna().unique())

view_mapping_summary = pd.DataFrame({
    "metric": [
        "view_unique_user_num",
        "mapping_unique_user_num",
        "view_user_num_in_mapping",
        "view_user_num_not_in_mapping",
    ],
    "value": [
        len(view_user_nums),
        len(mapping_user_nums),
        len(view_user_nums & mapping_user_nums),
        len(view_user_nums - mapping_user_nums),
    ],
})

display(view_mapping_summary)

## 1-10. Movie_Master와 영화 메타데이터 기본 구조 확인

영화 메타데이터는 세 파일로 나뉩니다.

1. `Movie_Master_v1.csv`: `MOVIE_NUM`과 영화 제목을 가진 기준 테이블
2. Wavve 크롤링 CSV: Wavve 플랫폼에서 수집한 장르, 국가, 관람등급, 러닝타임 등
3. KOBIS 보완 CSV: Wavve에서 찾지 못한 영화에 대해 KOBIS API로 보완한 정보

이 노트북에서는 메타데이터 통합을 수행하지 않고, 커버리지와 중복 가능성만 확인합니다. 실제 통합은 `03_movie_metadata_unification.ipynb`에서 수행합니다.

In [ ]:
movie_summary = pd.DataFrame({
    "dataset": ["movie_master", "wavve_meta", "kobis_meta"],
    "rows": [len(movie_master), len(wavve_meta), len(kobis_meta)],
    "unique_title_or_movie": [
        movie_master["MOVIE_NUM"].nunique(),
        wavve_meta["query_title"].nunique(dropna=True),
        kobis_meta["wavve_title"].nunique(dropna=True),
    ],
})

display(movie_summary)

print("Wavve query_title 중복 행 수:", int(wavve_meta["query_title"].duplicated(keep=False).sum()))
print("KOBIS wavve_title 중복 행 수:", int(kobis_meta["wavve_title"].duplicated(keep=False).sum()))

In [ ]:
def normalize_title(value) -> str:
    # 제목 매칭 검산용 간단 정규화 함수입니다. 본격 매칭 함수는 03번에서 별도로 관리합니다.
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[\[\]{}<>]", "", text)
    return text

movie_for_coverage = movie_master.copy()
movie_for_coverage["title_key"] = movie_for_coverage["movie_title"].map(normalize_title)

wavve_keys = set(wavve_meta["query_title"].dropna().map(normalize_title))
kobis_keys = set(kobis_meta["wavve_title"].dropna().map(normalize_title))

movie_for_coverage["covered_by_wavve"] = movie_for_coverage["title_key"].isin(wavve_keys)
movie_for_coverage["covered_by_kobis"] = movie_for_coverage["title_key"].isin(kobis_keys)
movie_for_coverage["covered_by_any"] = movie_for_coverage["covered_by_wavve"] | movie_for_coverage["covered_by_kobis"]

view_movie_nums = set(view_history["MOVIE_NUM"].dropna().unique())
view_movies_for_coverage = movie_for_coverage.loc[movie_for_coverage["MOVIE_NUM"].isin(view_movie_nums)].copy()

coverage_summary = pd.DataFrame([
    {
        "scope": "Movie_Master_all",
        "total_movies": len(movie_for_coverage),
        "wavve_covered": int(movie_for_coverage["covered_by_wavve"].sum()),
        "kobis_only_covered": int((~movie_for_coverage["covered_by_wavve"] & movie_for_coverage["covered_by_kobis"]).sum()),
        "any_covered": int(movie_for_coverage["covered_by_any"].sum()),
        "missing": int((~movie_for_coverage["covered_by_any"]).sum()),
        "coverage_rate": float(movie_for_coverage["covered_by_any"].mean()),
    },
    {
        "scope": "View_History_movies_only",
        "total_movies": len(view_movies_for_coverage),
        "wavve_covered": int(view_movies_for_coverage["covered_by_wavve"].sum()),
        "kobis_only_covered": int((~view_movies_for_coverage["covered_by_wavve"] & view_movies_for_coverage["covered_by_kobis"]).sum()),
        "any_covered": int(view_movies_for_coverage["covered_by_any"].sum()),
        "missing": int((~view_movies_for_coverage["covered_by_any"]).sum()),
        "coverage_rate": float(view_movies_for_coverage["covered_by_any"].mean()),
    },
])

display(coverage_summary)
coverage_summary.to_csv(TABLES_DIR / "01_data_overview_metadata_coverage.csv", index=False, encoding="utf-8-sig")

### 영화 메타데이터 커버리지에 대한 현재 판단

현재 데이터에서는 `View_History`에 등장하는 고유 영화 5,196편 중 약 4,575편이 Wavve 또는 KOBIS 메타데이터로 커버됩니다.

즉, 전체 `Movie_Master` 기준 커버율은 낮아 보일 수 있지만, 실제 시청이력에 등장하는 영화 기준으로는 약 88%가 커버됩니다.

이 때문에 영화 메타데이터는 버릴 필요가 없습니다. 다만 장르 단독 효과가 강하지 않았으므로, 이후에는 장르뿐 아니라 관람등급, 국가, 러닝타임, 개봉연식, 콘텐츠 최신성까지 포함한 유저 콘텐츠 성향 변수로 확장합니다.

## 1-11. KOBIS 매칭 위험성 예비 확인

KOBIS 보완 데이터는 Wavve에서 찾지 못한 영화를 보완하기 위한 자료입니다.

하지만 제목 기반 API 매칭은 동명이작 위험이 있습니다. 예를 들어 같은 제목의 영화가 여러 국가, 여러 장르, 여러 연도로 존재할 수 있습니다. 따라서 KOBIS 메타데이터는 그대로 맹신하지 않고, 03번 노트북에서 `v2` 통합 기준으로 품질 플래그를 추가합니다.

이 셀에서는 제목에 연도 힌트가 들어 있는 KOBIS 행과 KOBIS 제작연도가 어긋나는 사례를 간단히 확인합니다.

In [ ]:
def extract_year_hint(title) -> float:
    if pd.isna(title):
        return np.nan
    matches = re.findall(r"(?:19|20)\d{2}", str(title))
    if not matches:
        return np.nan
    return float(matches[-1])

kobis_check = kobis_meta.copy()
kobis_check["title_year_hint"] = kobis_check["wavve_title"].map(extract_year_hint)
kobis_check["prdtYear_numeric"] = pd.to_numeric(kobis_check["prdtYear"], errors="coerce")
kobis_check["year_diff"] = (kobis_check["title_year_hint"] - kobis_check["prdtYear_numeric"]).abs()

kobis_year_mismatch = kobis_check.loc[
    kobis_check["title_year_hint"].notna()
    & kobis_check["prdtYear_numeric"].notna()
    & (kobis_check["year_diff"] > 1)
].copy()

print("제목에 연도 힌트가 있고 KOBIS 제작연도와 1년 초과 차이 나는 행 수:", len(kobis_year_mismatch))

cols_to_show = ["wavve_title", "movieNm(api)", "prdtYear", "openDt", "nationNm", "genreNm", "watchGrade", "year_diff"]
display(kobis_year_mismatch[cols_to_show].head(20))

### KOBIS에 대한 현재 판단

KOBIS 파일은 이미 후보 중 하나가 선택된 999행짜리 보완 데이터입니다. 현재 파일만으로는 후보 목록 전체를 다시 비교할 수 없습니다.

따라서 이후 `03_movie_metadata_unification.ipynb`에서는 다음 정책을 사용합니다.

```text
Wavve 매칭: 우선 사용
KOBIS 매칭: 제목 유사도와 연도 힌트로 품질 점검
KOBIS low-confidence: CSV에는 남기되 콘텐츠 피처 생성에서는 제외
```

이 정책을 통해 영화 메타데이터의 커버율은 유지하되, 동명이작 오매칭으로 인한 장르, 국가, 관람등급 오염을 줄입니다.

## 1-12. 핵심 key 연결 가능성 요약

이 셀에서는 이후 파이프라인에서 사용할 주요 연결 관계를 간단히 검산합니다.

연결 구조는 다음과 같습니다.

```text
Membership.USER_KEY
  -> User_Mapping.USER_KEY
  -> User_Mapping.USER_NUM
  -> View_History.USER_NUM
  -> View_History.MOVIE_NUM
  -> Movie_Master.MOVIE_NUM
  -> Wavve/KOBIS metadata
```

In [ ]:
key_checks = []

mem_keys = set(membership["USER_KEY"].dropna().unique())
map_keys = set(user_mapping["USER_KEY"].dropna().unique())
key_checks.append({
    "check": "Membership.USER_KEY in User_Mapping.USER_KEY",
    "left_unique": len(mem_keys),
    "right_unique": len(map_keys),
    "matched_unique": len(mem_keys & map_keys),
    "left_only_unique": len(mem_keys - map_keys),
})

map_user_nums = set(user_mapping["USER_NUM"].dropna().unique())
view_user_nums = set(view_history["USER_NUM"].dropna().unique())
key_checks.append({
    "check": "View_History.USER_NUM in User_Mapping.USER_NUM",
    "left_unique": len(view_user_nums),
    "right_unique": len(map_user_nums),
    "matched_unique": len(view_user_nums & map_user_nums),
    "left_only_unique": len(view_user_nums - map_user_nums),
})

view_movie_nums = set(view_history["MOVIE_NUM"].dropna().unique())
master_movie_nums = set(movie_master["MOVIE_NUM"].dropna().unique())
key_checks.append({
    "check": "View_History.MOVIE_NUM in Movie_Master.MOVIE_NUM",
    "left_unique": len(view_movie_nums),
    "right_unique": len(master_movie_nums),
    "matched_unique": len(view_movie_nums & master_movie_nums),
    "left_only_unique": len(view_movie_nums - master_movie_nums),
})

key_checks_df = pd.DataFrame(key_checks)
display(key_checks_df)
key_checks_df.to_csv(TABLES_DIR / "01_data_overview_key_checks.csv", index=False, encoding="utf-8-sig")

## 1-13. 01번 노트북의 결론

이 노트북에서 확인한 핵심은 다음입니다.

| 항목 | 현재 판단 |
|---|---|
| 원본 CSV 로딩 | 정상 |
| `price == 100`과 `is_promotion == 1` 관계 | 현재 데이터에서는 완전히 일치 |
| 더미 이상치 후보 | `gender=N`, `is_user_verified=0`, `age=40` 조건 2,639건 확인 |
| User_Mapping 중복 | 존재함. 구독 이벤트 단위 분석에서는 오류로 단정하지 않음 |
| View_History 날짜 범위 | 2021-03-01부터 2021-04-05까지 |
| View_History 고유 영화 수 | 5,196편 |
| 영화 메타데이터 커버리지 | View_History 등장 영화 기준 약 88% |
| KOBIS 보완 데이터 | 유용하지만 동명이작 오매칭 위험이 있어 v2 품질 플래그 필요 |

따라서 다음 단계인 `02_preprocessing_policy.ipynb`에서는 다음 정책을 확정합니다.

1. 더미 이상치 제거 기준
2. 구독기간 21일 미만 케이스 처리
3. 고객별 `reg_date` 기준 day 0~20 관측창 정의
4. 시청이력 없는 고객을 모델링 테이블에 남길지 여부
5. `USER_KEY` 중복을 구독 이벤트 단위 분석에서 어떻게 해석할지

In [ ]:
# 이 노트북에서 생성된 감사 로그 파일 확인
output_files = [
    TABLES_DIR / "01_data_overview_file_summary.csv",
    TABLES_DIR / "01_data_overview_column_overview.csv",
    TABLES_DIR / "01_data_overview_key_checks.csv",
    TABLES_DIR / "01_data_overview_metadata_coverage.csv",
]

for path in output_files:
    print(path, "exists=", path.exists())